# 🔍 03 — Exploratory Data Analysis (EDA)

**Objective**: Discover patterns, trends, and relationships in the cleaned data through visualizations and statistical analysis.

**Why this step matters**: EDA reveals the story behind the data. It tells us which states drive revenue, how customers behave over time, what products are most popular, and where the business can improve — all critical for framing the right ML problem.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import set_plot_style, COLORS, PALETTE, format_currency

set_plot_style()
pd.set_option('display.float_format', '{:.2f}'.format)

# Load cleaned data
df = pd.read_csv('../data/master_cleaned.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns ✓")

## 1. Revenue Analysis

In [ ]:
# Total KPIs
total_revenue = df.drop_duplicates('order_id')['payment_value'].sum()
total_orders = df['order_id'].nunique()
total_customers = df['customer_unique_id'].nunique()
avg_order_value = total_revenue / total_orders

print("=" * 50)
print("  KEY PERFORMANCE INDICATORS")
print("=" * 50)
print(f"  Total Revenue:     {format_currency(total_revenue)}")
print(f"  Total Orders:      {total_orders:,}")
print(f"  Unique Customers:  {total_customers:,}")
print(f"  Avg Order Value:   {format_currency(avg_order_value)}")
print("=" * 50)

In [ ]:
# Revenue by State — Top 10
orders_unique = df.drop_duplicates('order_id')
rev_by_state = orders_unique.groupby('customer_state')['payment_value'].sum()\
    .sort_values(ascending=False).head(10).reset_index()
rev_by_state.columns = ['State', 'Revenue']

fig, ax = plt.subplots(figsize=(10, 6))
colors = [COLORS['primary'] if i > 0 else COLORS['success'] for i in range(len(rev_by_state))]
ax.barh(rev_by_state['State'][::-1], rev_by_state['Revenue'][::-1],
        color=colors[::-1], edgecolor='none', height=0.6)
ax.set_xlabel('Revenue (R$)')
ax.set_title('Top 10 States by Revenue', fontweight='bold', fontsize=14)
for i, v in enumerate(rev_by_state['Revenue'][::-1]):
    ax.text(v + total_revenue*0.005, i, f'R$ {v:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nSão Paulo alone accounts for {rev_by_state.iloc[0]['Revenue']/total_revenue*100:.1f}% of total revenue")

## 2. Monthly Revenue Trend

In [ ]:
# Monthly revenue trend
orders_unique = df.drop_duplicates('order_id').copy()
orders_unique['month'] = orders_unique['order_purchase_timestamp'].dt.to_period('M')
monthly_rev = orders_unique.groupby('month')['payment_value'].sum().reset_index()
monthly_rev['month_str'] = monthly_rev['month'].astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(range(len(monthly_rev)), monthly_rev['payment_value'],
                alpha=0.3, color=COLORS['primary'])
ax.plot(range(len(monthly_rev)), monthly_rev['payment_value'],
        color=COLORS['primary'], linewidth=2, marker='o', markersize=4)
ax.set_xticks(range(0, len(monthly_rev), 2))
ax.set_xticklabels(monthly_rev['month_str'].iloc[::2], rotation=45, ha='right')
ax.set_ylabel('Revenue (R$)')
ax.set_title('Monthly Revenue Trend', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Product Category Analysis

In [ ]:
# Top 10 categories by revenue
cat_rev = df.groupby('product_category_name_english')['price'].sum()\
    .sort_values(ascending=False).head(10).reset_index()
cat_rev.columns = ['Category', 'Revenue']

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(cat_rev)))
ax.barh(cat_rev['Category'][::-1], cat_rev['Revenue'][::-1],
        color=colors, edgecolor='none', height=0.6)
ax.set_xlabel('Revenue (R$)')
ax.set_title('Top 10 Product Categories by Revenue', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Payment Analysis

In [ ]:
# Payment method distribution
orders_unique = df.drop_duplicates('order_id')
pay_dist = orders_unique['payment_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = [COLORS['primary'], COLORS['secondary'], COLORS['success'], COLORS['warning']]
axes[0].pie(pay_dist.values, labels=pay_dist.index, autopct='%1.1f%%',
            colors=colors, startangle=90, pctdistance=0.85)
centre_circle = plt.Circle((0, 0), 0.60, fc='white')
axes[0].add_artist(centre_circle)
axes[0].set_title('Payment Method Distribution', fontweight='bold')

# Average installments by payment type
avg_inst = orders_unique.groupby('payment_type')['payment_installments'].mean().sort_values(ascending=False)
axes[1].bar(avg_inst.index, avg_inst.values, color=colors[:len(avg_inst)], edgecolor='none')
axes[1].set_ylabel('Average Installments')
axes[1].set_title('Avg Installments by Payment Type', fontweight='bold')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Review Score Analysis

In [ ]:
# Review score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
score_counts = df.drop_duplicates('order_id')['review_score'].value_counts().sort_index()
colors_bar = ['#ef4444', '#f59e0b', '#eab308', '#10b981', '#6366f1']
axes[0].bar(score_counts.index, score_counts.values, color=colors_bar, edgecolor='none')
axes[0].set_xlabel('Review Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Review Score Distribution', fontweight='bold')

# Average review by category (top 10)
cat_review = df.groupby('product_category_name_english')['review_score'].agg(['mean', 'count'])
cat_review = cat_review[cat_review['count'] >= 50].sort_values('mean', ascending=False).head(10)
axes[1].barh(cat_review.index[::-1], cat_review['mean'][::-1],
             color=COLORS['secondary'], edgecolor='none', height=0.6)
axes[1].set_xlabel('Avg Review Score')
axes[1].set_title('Top 10 Categories by Review Score', fontweight='bold')
axes[1].set_xlim(3, 5)

plt.tight_layout()
plt.show()

print(f"\nOverall average review score: {df['review_score'].mean():.2f} / 5.0")

## 6. Delivery Performance

In [ ]:
# Delivery analysis
orders = df.drop_duplicates('order_id').dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date']).copy()
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['is_late'] = orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']

late_pct = orders['is_late'].mean() * 100
avg_delivery = orders['delivery_days'].mean()

print(f"  Average delivery time: {avg_delivery:.1f} days")
print(f"  Late delivery rate: {late_pct:.1f}%")
print(f"  On-time deliveries: {(~orders['is_late']).sum():,} / {len(orders):,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(orders['delivery_days'], bins=40, color=COLORS['primary'],
             edgecolor='none', alpha=0.8)
axes[0].axvline(avg_delivery, color=COLORS['danger'], linestyle='--', label=f'Mean: {avg_delivery:.1f} days')
axes[0].set_xlabel('Delivery Days')
axes[0].set_title('Delivery Time Distribution', fontweight='bold')
axes[0].legend()

# Late delivery by state
late_by_state = orders.groupby('customer_state')['is_late'].mean().sort_values(ascending=False).head(10)
axes[1].barh(late_by_state.index[::-1], (late_by_state.values * 100)[::-1],
             color=COLORS['danger'], edgecolor='none', height=0.6)
axes[1].set_xlabel('Late Delivery %')
axes[1].set_title('Top 10 States by Late Delivery Rate', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Customer Behavior — Repeat Purchases

In [ ]:
# Repeat purchase analysis
customer_orders = df.drop_duplicates('order_id').groupby('customer_unique_id')['order_id'].nunique()
repeat_customers = (customer_orders > 1).sum()
total = len(customer_orders)
repeat_rate = repeat_customers / total * 100

print(f"  Total unique customers: {total:,}")
print(f"  Repeat customers:       {repeat_customers:,}")
print(f"  Repeat purchase rate:   {repeat_rate:.1f}%")

fig, ax = plt.subplots(figsize=(8, 5))
order_dist = customer_orders.value_counts().sort_index().head(10)
ax.bar(order_dist.index, order_dist.values, color=COLORS['primary'], edgecolor='none')
ax.set_xlabel('Number of Orders')
ax.set_ylabel('Number of Customers')
ax.set_title('Customer Order Frequency Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Correlation Analysis

In [ ]:
# Correlation matrix of numeric features
numeric_cols = ['price', 'freight_value', 'payment_value', 'payment_installments', 'review_score']
existing_cols = [c for c in numeric_cols if c in df.columns]
corr = df[existing_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, square=True)
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## Key EDA Findings

| Finding | Business Implication |
|---------|---------------------|
| São Paulo drives ~35% of revenue | Concentrate marketing in SP, explore growth in underserved states |
| Credit card = 73% of payments | Optimize checkout for credit card UX, offer installment incentives |
| Review scores are J-shaped (skewed positive) | Most customers are satisfied; focus on fixing the 1-2 star cases |
| Late delivery rate ~15-20% | Logistics improvement can directly reduce churn |
| Repeat purchase rate is low (~15%) | Major retention opportunity — validates the churn prediction model |
| Year-end seasonal spike | Plan inventory and marketing campaigns for Nov-Dec |

**Next Step**: In notebook 04, we'll engineer customer-level features for the churn prediction model.